# Phase 2 — Code-Aware Indexing Validation

Proves tree-sitter symbol extraction + the BM25 code index work.

**Run first:** `uv run python -m archaeologist.indexing.run`

The key idea: we chunk **by symbol** (class / method / function / import), so every search hit is a real code unit with a `file:line` you can open.

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from sqlalchemy import func, select
from archaeologist.models.db import session_scope
from archaeologist.models.entities import Symbol
from archaeologist.indexing import code_index
from archaeologist.indexing.opensearch_client import get_client
client = get_client()
print("repo root:", root)

## Symbols by kind (from Postgres)

In [ ]:
with session_scope() as s:
    total = s.scalar(select(func.count()).select_from(Symbol))
    print(f"total symbols: {total}\n")
    for kind, n in s.execute(select(Symbol.kind, func.count())
                             .group_by(Symbol.kind).order_by(func.count().desc())):
        print(f"  {kind:9}: {n}")

## Symbol-based chunking, seen on one file
The classes/methods tree-sitter pulled out of `src/flask/views.py`, with line ranges.

In [ ]:
with session_scope() as s:
    rows = s.scalars(select(Symbol)
                     .where(Symbol.file_path == "src/flask/views.py")
                     .order_by(Symbol.start_line)).all()
    for sym in rows:
        doc = (sym.docstring or "").splitlines()[0][:45] if sym.docstring else ""
        print(f"  L{sym.start_line:<4}-{sym.end_line:<4} [{sym.kind:8}] {sym.qualified_name:32.32} {doc}")

## BM25 search — natural-language questions → code units
Every hit carries a `file:line`. This is the keyword foundation the course builds vectors on top of next.

In [ ]:
QUERIES = [
    "where is url routing dispatched to view functions",
    "how are before_request handlers registered and run",
    "json serialization of responses",
]
for q in QUERIES:
    print(f"\n=== {q!r} ===")
    for hit in code_index.search(client, q, k=4):
        loc = f"{hit['file_path']}:{hit['start_line']}"
        print(f"  {hit['score']:6.2f}  [{hit['kind']:8}] {hit['qualified_name']:38.38} {loc}")

## Filtered search — classes only

In [ ]:
for hit in code_index.search(client, "application and blueprint objects", k=5, kind="class"):
    loc = f"{hit['file_path']}:{hit['start_line']}"
    print(f"  {hit['score']:6.2f}  {hit['qualified_name']:28.28} {loc}")

## Summary

In [ ]:
with session_scope() as s:
    pg = s.scalar(select(func.count()).select_from(Symbol))
os_count = client.count(index=code_index.SYMBOL_INDEX)["count"]
ok = pg > 0 and pg == os_count
print("Phase 2 —", "INDEXING OK ✅" if ok else "MISMATCH ❌")
print(f"  symbols in Postgres : {pg}")
print(f"  docs in OpenSearch  : {os_count}")